In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 1. Load sample dataset
data = load_iris()
X, y = data.data, data.target

# 2. Define the Fitness Function (Objective Function)
def get_fitness(hyperparams):
    """
    Evaluates a specific set of hyperparameters using 3-fold cross-validation.
    GWO natively works with continuous spaces, so we cast to integer boundaries.
    """
    n_estimators = int(np.clip(hyperparams[0], 10, 200))
    max_depth = int(np.clip(hyperparams[1], 2, 20))

    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    # We negate or use raw accuracy depending on if we are minimizing or maximizing.
    # GWO conventionally minimizes an error function. We will minimize (1 - accuracy).
    scores = cross_val_score(clf, X, y, cv=3, n_jobs=-1)
    return 1.0 - np.mean(scores)

# 3. Grey Wolf Optimizer for Hyperparameters
def grey_wolf_optimize(fitness_func, num_dimensions, lb, ub, num_wolves=10, max_iter=15):
    # Initialize positions of wolves randomly within bounds
    positions = np.random.uniform(0, 1, (num_wolves, num_dimensions)) * (ub - lb) + lb

    # Initialize Alpha, Beta, and Delta structures
    alpha_pos = np.zeros(num_dimensions)
    alpha_score = float('inf')

    beta_pos = np.zeros(num_dimensions)
    beta_score = float('inf')

    delta_pos = np.zeros(num_dimensions)
    delta_score = float('inf')

    # Main Optimization Loop
    for t in range(max_iter):
        for i in range(num_wolves):
            # Check boundary constraints
            positions[i] = np.clip(positions[i], lb, ub)

            # Calculate fitness for the current wolf
            fitness = fitness_func(positions[i])

            # Update Alpha, Beta, and Delta hierarchies
            if fitness < alpha_score:
                alpha_score = fitness
                alpha_pos = positions[i].copy()
            elif fitness < beta_score:
                beta_score = fitness
                beta_pos = positions[i].copy()
            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = positions[i].copy()

        # Linearly decrease 'a' from 2 to 0 to balance exploration and exploitation
        a = 2 - t * (2 / max_iter)

        # Update the position of all omega wolves
        for i in range(num_wolves):
            for j in range(num_dimensions):
                # Update equations regarding Alpha leader
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                # Update equations regarding Beta leader
                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                # Update equations regarding Delta leader
                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                # The new position is the average of the three main trajectory steps
                positions[i, j] = (X1 + X2 + X3) / 3

        print(f"Iteration {t+1}/{max_iter} - Best Error Rate (Alpha): {alpha_score:.4f}")

    return alpha_pos, alpha_score

# 4. Define Search Boundaries
# Dimension 0: n_estimators (Range: 10 to 200)
# Dimension 1: max_depth (Range: 2 to 20)
lower_bounds = np.array([10, 2])
upper_bounds = np.array([200, 20])

best_params, best_error = grey_wolf_optimize(
    fitness_func=get_fitness,
    num_dimensions=2,
    lb=lower_bounds,
    ub=upper_bounds,
    num_wolves=8,
    max_iter=5
)

print("\n--- Optimization Complete ---")
print(f"Optimal n_estimators: {int(best_params[0])}")
print(f"Optimal max_depth: {int(best_params[1])}")
print(f"Best Validation Accuracy achieved: {1.0 - best_error:.4f}")


Iteration 1/5 - Best Error Rate (Alpha): 0.0333
Iteration 2/5 - Best Error Rate (Alpha): 0.0333
Iteration 3/5 - Best Error Rate (Alpha): 0.0333
Iteration 4/5 - Best Error Rate (Alpha): 0.0333
Iteration 5/5 - Best Error Rate (Alpha): 0.0333

--- Optimization Complete ---
Optimal n_estimators: 130
Optimal max_depth: 18
Best Validation Accuracy achieved: 0.9667
